In [ ]:

# %pip install langgraph langchain-core langchain-openai
# %pip install rich python-dotenv
# %pip install tavily-python langchain-tavily

### Defining state

 - Think of this as the "contract" for your entire pipeline.
 - Every node can read ANY field, and return updates to ANY field.

 - Why TypedDict and not a regular dict?
    - It gives you type hints, autocomplete, and catches typos.
    - LangGraph uses it to understand what your state looks like.

In [1]:
from typing import TypedDict, Optional

class ContentState(TypedDict):
    topic: str                          # Input: what to write about
    research_notes: str                 # Filled by Researcher (later)
    draft: str                          # Filled by Writer
    review_score: Optional[int]         # Filled by Reviewer (0-10)
    feedback: str                       # Filled by Reviewer

In [3]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()  

api_key = os.getenv("OPEN_ROUTER_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")


llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free", 
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:3000", # Required by OpenRouter for some models
        "X-Title": "LangGraph App",
    }
)

### Researcher

In [6]:

from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

def researcher_node(state: ContentState) -> dict:
    """
    The Researcher node:
    - Reads:  'topic' from state
    - Writes: 'research_notes' to state
    
    How it works:
    1. LLM with tools decides WHAT to search
    2. We execute the search
    3. LLM synthesizes results into clean research notes
    
    This is the "ReAct" pattern:
    Reason → Act (use tool) → Observe results → Reason again
    """
    print("\n🔍 RESEARCHER NODE ACTIVATED")
    
    
    topic = state["topic"]
    
    # ── Step A: Set up LLM with Tool ─────────────────────────
    search_tool = TavilySearch(max_results=5)
    llm_with_tools = llm.bind_tools([search_tool])
    
    # ── Step B: Ask the LLM what to search for ───────────────
    print(f"📌 Topic to research: {topic}")
    
    messages = [
        SystemMessage(content="""You are a research assistant. 
Your job is to search for information about the given topic.
Use the search tool to find relevant, up-to-date information."""),
        
        HumanMessage(content=f"""Research this topic thoroughly: {topic}
        
Please search for:
1. Key concepts and definitions
2. Current trends and developments  
3. Important facts and statistics
4. Expert opinions or analysis

Use the search tool now."""),
    ]
    
    # First LLM call: LLM decides to call the tool
    print("🤖 LLM deciding what to search for...")
    llm_response = llm_with_tools.invoke(messages)
    
    print(f"🔧 Tool calls requested: {len(llm_response.tool_calls)}")
    
    # ── Step C: Execute the Tool Calls ───────────────────────
    # The LLM told us WHAT to search — now we actually DO the search
    
    all_search_results = []  # Collect all raw results here
    tool_messages = []       # These go back to the LLM for synthesis
    
    for tool_call in llm_response.tool_calls:
        print(f"🌐 Executing search: '{tool_call['args'].get('query', 'N/A')}'")
        
        # Actually run the Tavily search
        search_results = search_tool.invoke(tool_call["args"])
        
        # search_results is a list of dicts with keys:
        # 'url', 'content', 'title', 'score'
        all_search_results.append(search_results)
        
        print(f"   ✅ Got {len(search_results)} results")
        
        # Wrap results in a ToolMessage so the LLM can read them
        # The tool_call_id links this result back to the specific tool call
        tool_messages.append(
            ToolMessage(
                content=str(search_results),
                tool_call_id=tool_call["id"],
            )
        )
    
    # ── Step D: LLM Synthesizes the Results ──────────────────
    # Now feed the raw search results BACK to the LLM
    # Ask it to write clean, structured research notes
    
    print("📝 LLM synthesizing search results into research notes...")
    
    # Build the full conversation: original messages + LLM response + tool results
    synthesis_messages = messages + [llm_response] + tool_messages + [
        HumanMessage(content=f"""Based on these search results, write comprehensive 
research notes about: {topic}

Structure your notes as:
## Key Findings
[main points discovered]

## Important Facts & Data  
[specific statistics, dates, numbers]

## Current Trends
[what's happening now]

## Summary
[2-3 sentence overview]

Be specific and cite the sources where relevant.""")
    ]
    
    # Second LLM call: pure synthesis, no more tool calls needed
    final_response = llm.invoke(synthesis_messages)
    
    research_notes = final_response.content
    
    print(f"✅ Research complete! Notes: {len(research_notes)} characters")
    print("─" * 50)
    
    return {"research_notes": research_notes}

### The writer

In [7]:
def writer_node(state: ContentState) -> dict:
    """
    Writer node:
    - Reads:  'topic', 'research_notes', and optionally 'feedback'
    - Writes: 'draft'
    """
    print("\n✍️  WRITER NODE ACTIVATED")
    
    topic          = state["topic"]
    research_notes = state["research_notes"]
    feedback       = state.get("feedback", "")
    previous_draft = state.get("draft", "")
    
    # ── Decide which prompt to use ────────────────────────────
    # First time writing vs. revision based on reviewer feedback
    
    if feedback and previous_draft:
        # ── REVISION PATH: Reviewer sent it back ─────────────
        print("🔄 Revision mode: incorporating reviewer feedback...")
        
        prompt = f"""You are a professional content writer doing a REVISION.

                TOPIC: {topic}

                RESEARCH NOTES (use these facts):
                {research_notes}

                YOUR PREVIOUS DRAFT:
                {previous_draft}

                REVIEWER FEEDBACK TO ADDRESS:
                {feedback}

                Instructions:
                - Keep what worked well in the previous draft
                - Specifically address each piece of feedback
                - Use the research notes to add accuracy and depth
                - Write 3-4 engaging paragraphs
                - Do NOT mention that this is a revision

                Write the improved article now:"""

    else:
        # ── FIRST DRAFT PATH ──────────────────────────────────
        print("✨ First draft mode: writing from research notes...")
        
        prompt = f"""You are a professional content writer.

                TOPIC: {topic}

                RESEARCH NOTES (these are your source material — use them!):
                {research_notes}

                Instructions:
                - Base your article on the research notes above
                - Write an engaging introduction that hooks the reader
                - Cover the key findings and important facts
                - Include specific data points and trends from the research
                - Write a compelling conclusion
                - Aim for 3-4 well-structured paragraphs
                - Write for a general but intelligent audience

                Write the article now:"""
    
    print("🤖 LLM writing the draft...")
    response = llm.invoke(prompt)
    draft = response.content
    
    print(f"✅ Draft complete! ({len(draft)} characters)")
    print("─" * 50)
    
    return {"draft": draft}

### Reviewer node

In [8]:
def reviewer_node(state: ContentState) -> dict:
    """
    The Reviewer node:
    - Reads: 'draft' and 'topic' from state  
    - Writes: 'review_score' and 'feedback' to state
    
    This uses the LLM to actually evaluate the draft.
    """
    print("🔍 Reviewer node activated...")
    
    draft = state["draft"]
    topic = state["topic"]
    
    prompt = f"""You are a strict content reviewer. 
    
    Review this draft about "{topic}":

    ---
    {draft}
    ---

    Evaluate on these criteria:
    1. Accuracy and relevance to the topic
    2. Clarity and readability  
    3. Engagement and flow
    4. Completeness

    Respond in EXACTLY this format (no extra text):
    SCORE: [number from 1-10]
    FEEDBACK: [your specific feedback for improvement]"""

    response = llm.invoke(prompt)
    response_text = response.content
    
    # Parse the score and feedback from the response
    try:
        lines = response_text.strip().split("\n")
        score_line = [l for l in lines if l.startswith("SCORE:")][0]
        score = int(score_line.split(":")[1].strip().split("/")[0].split()[0])
        
        feedback_start = response_text.find("FEEDBACK:")
        feedback = response_text[feedback_start + 9:].strip() if feedback_start != -1 else "No specific feedback."
    except (ValueError, IndexError):
        # If parsing fails, default to a middle score
        score = 5
        feedback = response_text
    
    # Clamp score to valid range
    score = max(1, min(10, score))
    
    print(f"🔍 Review complete — Score: {score}/10")
    print(f"🔍 Feedback: {feedback[:100]}...")
    
    return {
        "review_score": score,
        "feedback": feedback,
    }

### Building blocks

In [9]:
from langgraph.graph import StateGraph, START, END

# Build the graph
graph_builder = StateGraph(ContentState)

# ── Register all nodes ────────────────────────────────────
graph_builder.add_node("researcher", researcher_node) 
graph_builder.add_node("writer",     writer_node)
graph_builder.add_node("reviewer",   reviewer_node)

# ── Define the edges (execution order) ───────────────────
#
# Flow: START → researcher → writer → reviewer → END
#
graph_builder.add_edge(START,        "researcher")  
graph_builder.add_edge("researcher", "writer")      
graph_builder.add_edge("writer",     "reviewer")
graph_builder.add_edge("reviewer",   END)

# ── Compile ───────────────────────────────────────────────
graph = graph_builder.compile()

print("✅ Updated graph compiled successfully!")
print("\nExecution order:")
print("  START → researcher → writer → reviewer → END")

✅ Updated graph compiled successfully!

Execution order:
  START → researcher → writer → reviewer → END


### Visualize the graph

In [ ]:
# LangGraph can render your graph as an image — super useful for debugging!

from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # If the image rendering fails, print the text version
    print("Graph structure (text):")
    graph.get_graph().print_ascii()
    print(f"\n(Image rendering requires additional deps: {e})")

### Run

In [11]:
initial_state = {
    "topic": "The impact of AI agents on software engineering in 2026",
    "research_notes": "",    # Researcher will fill this
    "draft": "",             # Writer will fill this
    "review_score": None,    # Reviewer will fill this
    "feedback": "",          # Reviewer will fill this
}

print("🚀 STARTING AUTONOMOUS CONTENT ENGINE")
print("=" * 60)

# Use stream() to watch each node execute in real time
final_state = None

for event in graph.stream(initial_state, stream_mode="values"):
    # stream_mode="values" gives us the FULL state after each node
    # (vs default mode which gives partial updates per node)
    final_state = event  # Keep track of the latest state

print("\n" + "=" * 60)
print("🏁 PIPELINE COMPLETE!")

🚀 STARTING AUTONOMOUS CONTENT ENGINE

🔍 RESEARCHER NODE ACTIVATED
📌 Topic to research: The impact of AI agents on software engineering in 2026
🤖 LLM deciding what to search for...
🔧 Tool calls requested: 1
🌐 Executing search: 'AI agents software engineering 2026 impact'
   ✅ Got 7 results
📝 LLM synthesizing search results into research notes...
✅ Research complete! Notes: 6045 characters
──────────────────────────────────────────────────

✍️  WRITER NODE ACTIVATED
✨ First draft mode: writing from research notes...
🤖 LLM writing the draft...
✅ Draft complete! (3011 characters)
──────────────────────────────────────────────────
🔍 Reviewer node activated...
🔍 Review complete — Score: 5/10
🔍 Feedback: The draft presents an engaging vision but relies on unverified statistics (e.g., Claude Code commit ...

🏁 PIPELINE COMPLETE!


In [12]:
# ============================================================
# INSPECT THE FINAL STATE
# ============================================================
from rich import print as rprint

rprint(f"\n📌 Topic: {final_state['topic']}")
rprint(f"\n📝 Final Draft:\n{final_state['draft']}")
rprint(f"\n⭐ Review Score: {final_state['review_score']}/10")
rprint(f"\n💬 Feedback:\n{final_state['feedback']}")

📌 Topic: The impact of AI agents on software engineering in 2026

📝 Final Draft:
**The software factory of 2026 no longer looks like rows of developers hunched over keyboards; it resembles a 
control room where human engineers direct fleets of AI agents that write, test, and ship code at unprecedented 
speed.** This shift from solitary coding to AI‑orchestrated development is not a distant vision—it is already 
reflected in the data flowing from GitHub, cloud databases, and enterprise surveys. In early 2026, Claude Code 
accounted for roughly 4 % of all GitHub commits, a figure projected to surpass 20 % by year‑end, while AI agents 
generated 80 % of new Neon databases and pushed TypeScript past Python as the top language on GitHub, thanks to the
type‑safety that agents favor.  

The net effect is a step‑function lift in output: engineers spend less time on any single task but ship far more 
features, bug fixes, and experiments. Development cycles that once stretched weeks now collapse into days, turning 
previously marginal projects into viable bets and boosting ROI through faster time‑to‑value. Yet the surge in 
AI‑generated code brings a new kind of friction—“AI slop.” Low‑quality, auto‑produced snippets increase review 
burdens, inject bugs, and leave some senior builders feeling displaced from the hands‑on craft they once enjoyed.  

Technically, today’s agents are far more sophisticated than simple autocomplete tools. Models such as 
Qwen3‑Coder‑Next (80 B parameters), Claude Code, and Codex now possess repository‑level understanding, embed 
security scanning, and can autonomously handle feasibility analysis, scaffolding, implementation, testing, and 
documentation across the full SDLC. However, reliability compounds risk: a step that is 95 % reliable yields only 
about a 36 % chance of success after twenty chained actions, underscoring why fewer than 10 % of firms running 
agents in production can truly govern them. Regulatory pressure is mounting, with the EU AI Act’s Article 50 
deadline looming on 2 August 2026 and U.S. state laws like California’s SB 53 and New York’s RAISE Act close 
behind.  

Beyond the engineering team, the ripple is economic. Agent‑driven automation has eroded traditional per‑seat SaaS 
pricing, triggering a “SaaSpocalypse” that wiped roughly $2 trillion of market capitalization in early 2026 as 
dozens of licensed seats were replaced by autonomous services. Organizations are responding by shifting value 
toward agent‑consumption or outcome‑based models, while demanding new skill sets: orchestration, systems thinking, 
and strategic product definition now outweigh pure syntax prowess. As engineers curate agent output, focus on 
architecture, and steer multi‑agent workflows, the profession is being reframed—not as a decline of coding, but as 
an elevation to higher‑order stewardship of software creation. The result is a faster, leaner pipeline that 
promises tremendous upside, provided companies invest in governance, quality safeguards, and the human‑AI 
collaboration skills that will define the next era of software engineering.

⭐ Review Score: 5/10

💬 Feedback:
The draft presents an engaging vision but relies on unverified statistics (e.g., Claude Code commit share, Neon 
database generation) that undermine accuracy. Replace speculative numbers with cited, verifiable data or clearly 
label them as projections. Improve flow by tightening transitions between sections (e.g., from technical 
reliability to economic impact) and add concrete examples or case studies to illustrate points. Ensure all claims 
about regulatory timelines and market effects are backed by sources. Finally, proofread for minor grammatical slips
and vary sentence length to enhance readability.